In [ ]:
import sys
!{sys.executable} -m pip show youtube-transcript-api
!{sys.executable} -m pip install -U youtube-transcript-api

In [12]:
# 유튜브 사이트 크롤링
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.common.exceptions import WebDriverException
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.chrome.options import Options
import undetected_chromedriver as uc
import time , tempfile 
import sys   
import math   
import pandas as pd   
import os
import random
import urllib.request
import urllib
from openpyxl.styles import Font
from openpyxl import load_workbook
from openpyxl.drawing.image import Image as XLImage
import chromedriver_autoinstaller
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import (
    TranscriptsDisabled, NoTranscriptFound, VideoUnavailable
)
import re

In [ ]:
print('='*80)
print('< 이 프로그램은 유튜브 데이터를 수집하는 프로그램 입니다.>')
print('='*80)

qu = 'https://www.youtube.com/'

chromedriver_autoinstaller.install(path='c:/py_temp')
driver = webdriver.Chrome()

wait = WebDriverWait(driver, 10)

driver.get(qu)
time.sleep(2)
driver.maximize_window()
time.sleep(2)

now = time.localtime()
s = '%04d-%02d-%02d-%02d-%02d-%02d' %(now.tm_year, now.tm_mon, now.tm_mday, \
                                      now.tm_hour, now.tm_min, now.tm_sec)

ff = 'c:\\py_temp\\2차프로젝트\\'
if ff == '':
    ff = 'c:\\py_temp\\2차프로젝트\\'

img_dir = ff + s + '-' + '유튜브_애니성지순례' + '\\img'

ft_name = ff+s+'-'+'유튜브_애니성지순례'+'\\'+s+'-'+'유튜브_애니성지순례'+'.txt'
fx_name = ff+s+'-'+'유튜브_애니성지순례'+'\\'+s+'-'+'유튜브_애니성지순례'+'.xlsx'
fc_name = ff+s+'-'+'유튜브_애니성지순례'+'\\'+s+'-'+'유튜브_애니성지순례'+'.csv'

# ============================================
# ✅ YouTube 성지순례 영상 수집 + (가능하면) 자막 텍스트만 저장
# - selenium으로 검색 → 영상 링크 수집 → 영상 방문
# - 제목/채널/조회수/날짜/설명/댓글1개(가능하면) + 자막(가능하면) 저장
# - 자막 없는 영상은 "스크립트 지원 불가" 처리하고 계속 진행
# ============================================

import os
import re
import time
import urllib.parse

from bs4 import BeautifulSoup

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled, NoTranscriptFound, VideoUnavailable
from selenium.common.exceptions import TimeoutException, WebDriverException


# =========================
# ✅ 저장 리스트들
# =========================
no2 = []              # 번호
title2 = []           # 제목
name2 = []            # 닉네임(채널)
view2 = []            # 조회수
date2 = []            # 날짜
write2 = []           # 내용(설명)
reaction2 = []        # 1번 댓글(가능하면)
url2 = []             # 링크
transcript_ok2 = []   # 자막 추출 성공 여부(True/False)
transcript2    = []   # 자막 텍스트(없으면 None)
anime2   = []   # ✅ 애니 이름(시리즈)

# =========================
# ✅ (선택) 폴더 준비
# img_dir 변수가 이미 위에서 정의되어 있다고 가정
# =========================
os.makedirs(img_dir, exist_ok=True)
os.chdir(img_dir)
    
# =========================
# 0) 천천히 스크롤
# =========================
def scroll_down(driver, step=1800, sleep=4.0):
    driver.execute_script(f"window.scrollBy(0,{step});")
    time.sleep(4)


# =========================
# 1) "동영상" 칩 클릭 (텍스트 기반 + JS 클릭)
# =========================
def click_chip(driver, text="동영상", timeout=10):
    wait = WebDriverWait(driver, timeout)
    chip_btn_xpath = (
        f"//yt-chip-cloud-chip-renderer"
        f"//button[.//*[normalize-space()='{text}'] or .//div[normalize-space()='{text}']]"
    )
    btn = wait.until(EC.presence_of_element_located((By.XPATH, chip_btn_xpath)))
    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
    driver.execute_script("arguments[0].click();", btn)
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "ytd-video-renderer")))
    time.sleep(3)


# =========================
# 2) 검색 결과 페이지로 이동 (URL 직접 생성)
# =========================
def go_search_results(driver, keyword):
    url = "https://www.youtube.com/results?search_query=" + urllib.parse.quote(keyword)
    driver.get(url)

    wait = WebDriverWait(driver, 15)
    wait.until(lambda d: d.execute_script("return document.readyState") == "complete")
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "ytd-video-renderer")))
    return url


# =========================
# 3) 검색결과에서 링크 수집
# =========================
def collect_video_links(driver, target_count=40, max_steps=400, step_px=2000, sleep=3, seen_video_ids=None):
    wait = WebDriverWait(driver, 15)
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "ytd-video-renderer")))

    if seen_video_ids is None:
        seen_video_ids = set()

    links = []
    stable = 0
    last_len = 0

    dup_count = 0      # ✅ 중복 스킵 수
    new_count = 0      # ✅ 신규 수집 수

    for step in range(max_steps):
        elems = driver.find_elements(By.CSS_SELECTOR, "ytd-video-renderer a#thumbnail")
        for e in elems:
            href = e.get_attribute("href")
            if not href or "watch?v=" not in href:
                continue

            vid = get_video_id(href)
            if not vid:
                continue

            # ✅ 전역 중복 제거
            if vid in seen_video_ids:
                dup_count += 1
                continue

            seen_video_ids.add(vid)
            links.append(href)
            new_count += 1

            if len(links) >= target_count:
                print(f"  ▶ 신규 수집: {new_count}개 | 중복 스킵: {dup_count}개")
                return links[:target_count]

        # 증가 체크
        if len(links) == last_len:
            stable += 1
        else:
            stable = 0
            last_len = len(links)

        if stable >= 10:
            break

        driver.execute_script("window.scrollBy(0, arguments[0]);", step_px)
        time.sleep(sleep)

        try:
            last = driver.find_elements(By.CSS_SELECTOR, "ytd-video-renderer")[-1]
            driver.execute_script("arguments[0].scrollIntoView({block:'end'});", last)
            time.sleep(0.6)
        except:
            pass

    print(f"  ▶ 신규 수집: {new_count}개 | 중복 스킵: {dup_count}개")
    return links[:target_count]

# =========================
# 5) 중복영상 추출
# =========================
def get_video_id(url: str):
    if not isinstance(url, str):
        return None
    m = re.search(r"(?:v=|youtu\.be/|embed/)([A-Za-z0-9_-]{11})", url)
    return m.group(1) if m else None

# 댓글 클리너(간략히/자세히 보기 제거)
UI_LINES = {
    "간략히",
    "자세히 보기",
    "자세히보기",
    "더보기",
    "Show more",
    "Show less",
    "Read more",
    "Read less",
}

def clean_comment_text(text: str) -> str | None:
    if not text:
        return None

    # 1) 앞뒤 공백 제거
    t = text.strip()

    # 2) 줄 단위 분해 후, UI 문구 줄 제거 + 빈줄 제거
    lines = [ln.strip() for ln in t.splitlines()]
    lines = [ln for ln in lines if ln and ln not in UI_LINES]

    # 3) 다시 합치기 (원하면 "\n" 유지 / " "로 한 줄 만들기 선택 가능)
    t = " ".join(lines)

    # 4) 혹시 남는 과한 공백/줄바꿈 정리
    t = re.sub(r"\n{2,}", "\n", t)      # 연속 줄바꿈 2개 이상 -> 1개
    t = re.sub(r"[ \t]{2,}", " ", t)    # 연속 공백 -> 1칸

    return t.strip() if t.strip() else None

# =========================
# 6) 링크 방문 + 데이터 저장 (중복 호출 없음)
# =========================
def visit_each_video(driver, links, back_to_url, ani, no_start=1, dwell=2.0):
    wait = WebDriverWait(driver, 15)
    no = no_start

    for i, url in enumerate(links, 1):
        print(f"\n  [{i}/{len(links)}] 방문: {url}")
        # 0) 기본값(실패해도 리스트 길이 유지)
        title1 = name1 = view1 = date1 = write1 = reaction1 = None
        ok_t = False
        transcript_text = None

        driver.get(url)
        wait.until(lambda d: d.execute_script("return document.readyState") == "complete")
        time.sleep(1.0)

        # 설명 더보기 클릭 시도(실패해도 계속)
        try:
            scroll_down(driver, step=800, sleep=0.7)
            more_btn = driver.find_element(By.XPATH, "//tp-yt-paper-button[@id='expand']")
            driver.execute_script("arguments[0].click();", more_btn)
            time.sleep(0.5)
        except:
            pass

        # 페이지 파싱
        html = driver.page_source
        s = BeautifulSoup(html, "html.parser")

        # watch root 찾기(없으면 안전하게 None 처리)
        c1 = s.find("ytd-watch-flexy")

        # 번호
        no2.append(no)
        print("1. 번호:", no)
        
        # ✅ 여기 추가: 애니
        anime2.append(ani)
        print('애니 시리즈: ' + ani)

        # 제목
        try:
            title1 = c1.find("h1", "style-scope ytd-watch-metadata").get_text().strip()
        except:
            title1 = None
        title2.append(title1)
        print("2. 제목:", title1 if title1 else "(없음)")

        # 닉네임(채널)
        try:
            name1 = c1.find("a", "yt-simple-endpoint style-scope yt-formatted-string").get_text().strip()
        except:
            name1 = None
        name2.append(name1)
        print("3. 닉네임:", name1 if name1 else "(없음)")

        # 조회수 / 날짜 (레이아웃 바뀌면 안전하게 None)
        view1, date1 = None, None
        try:
            info = c1.find("div", "style-scope ytd-watch-info-text")
            spans = info.find_all("span")
            # 예전 네 방식 유지(깨지면 except로 빠짐)
            view1 = spans[0].get_text().strip()[4:]
            date1 = spans[2].get_text().strip()
        except:
            pass
        view2.append(view1)
        date2.append(date1)
        print("4. 조회수:", view1 if view1 else "(없음)")
        print("5. 날짜:", date1 if date1 else "(없음)")

        # 내용(설명)
        try:
            write1 = c1.find("span", "yt-core-attributed-string--link-inherit-color").get_text().strip()
        except:
            write1 = None
        write2.append(write1)
        print("6. 내용:", write1 if write1 else "내용이 없습니다.")

        # 1번 댓글(있으면)
        reaction1 = None
        try:
            raw = c1.find("ytd-expander", "style-scope ytd-comment-view-model") \
                    .get_text("\n", strip=True)   # ✅ 줄바꿈 유지해서 가져오기
            reaction1 = clean_comment_text(raw)
        except:
            reaction1 = None
        reaction2.append(reaction1)
        print("7. 1번댓글:", reaction1 if reaction1 else "내용이 없습니다.")

        # 링크
        url2.append(url)
        print("8. 링크:", url)
        print("=" * 90)

        no += 1
        time.sleep(dwell)

        # 검색결과로 복귀
        driver.get(back_to_url)
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "ytd-video-renderer")))
        time.sleep(1.0)

    return no  # 다음 번호 이어서 쓰기용


# =========================
# 7) 전체 루프: 8개 애니 순회
# =========================
animes = [
    "날씨의아이",
    "너의이름은",
    "스즈메의문단속",
    "주술회전",
    "체인소맨",
    "귀멸의 칼날",
    "명탐정코난",
    "슬램덩크",
]
no = 1

seen_video_ids = set()  # ✅ 전체 애니 공통 중복 제거용

for ani in animes:
    keyword = f"{ani} 성지순례"
    print("\n==============================")
    print("검색 키워드:", keyword)
    print("==============================")

    # 1) 검색 결과 진입
    results_url = go_search_results(driver, keyword)

    # 2) "동영상" 탭 클릭(실패해도 계속)
    try:
        click_chip(driver, "동영상")
        results_url = driver.current_url
    except Exception as e:
        print("  ⚠️ 동영상 칩 클릭 실패(그래도 진행):", type(e).__name__)

    # 3) 링크 수집 (✅ 중복 제거 포함)
    links = collect_video_links(
        driver,
        target_count=40,
        max_steps=400,
        step_px=2000,
        sleep=3,
        seen_video_ids=seen_video_ids
    )
    print("  수집 링크:", len(links))

    # 4) 링크 방문 + 저장
    no = visit_each_video(
    driver, links,
    back_to_url=results_url,
    ani=ani,
    no_start=no,
    dwell=2.0
    )


print("\n✅ 8개 애니 전체 루프 완료")

# =========================
# DataFrame 생성
# =========================
youtu = pd.DataFrame({
    "번호": no2,
    "애니": anime2,          # ✅ 추가
    "제목": title2,
    "닉네임": name2,
    "조회수": view2,
    "날짜": date2,
    "내용": write2,
    "1번 댓글": reaction2,
    "영상 링크": url2
})

# CSV 저장
youtu.to_csv(fc_name, encoding="utf-8-sig", index=False)

# =========================
# Excel 저장 (가독성 개선)
# =========================
with pd.ExcelWriter(fx_name, engine="xlsxwriter") as writer:
    youtu.to_excel(writer, index=False, sheet_name="video")

    wb = writer.book
    ws = writer.sheets["video"]

    # ---------- 스타일 ----------
    header_fmt = wb.add_format({
        "bold": True,
        "align": "center",
        "valign": "vcenter",
        "border": 1
    })

    cell_fmt = wb.add_format({
        "valign": "top",
        "border": 1
    })

    wrap_fmt = wb.add_format({
        "valign": "top",
        "text_wrap": True,
        "border": 1
    })

    # ---------- 헤더 스타일 ----------
    for col_idx, col_name in enumerate(youtu.columns):
        ws.write(0, col_idx, col_name, header_fmt)

    # ---------- 열 너비 설정 ----------
    col_widths = {
        "번호": 6,
        "애니": 14,          # ✅ 추가
        "제목": 35,
        "닉네임": 20,
        "조회수": 12,
        "날짜": 12,
        "내용": 50,
        "1번 댓글": 45,
        "영상 링크": 45
    }

    for col_name, width in col_widths.items():
        col_idx = youtu.columns.get_loc(col_name)
        ws.set_column(col_idx, col_idx, width)

    # ---------- 데이터 셀 스타일 ----------
    for r in range(1, len(youtu) + 1):
        for c, col_name in enumerate(youtu.columns):
            val = youtu.iloc[r - 1, c]

            # 링크 컬럼은 하이퍼링크 유지
            if col_name == "영상 링크" and isinstance(val, str) and val.strip():
                ws.write_url(r, c, val, string=val)
            else:
                fmt = wrap_fmt if col_name in ["내용", "1번 댓글"] else cell_fmt
                ws.write(r, c, val, fmt)

print("=" * 80)
print("엑셀 저장 완료 (가독성 개선 적용)")
print("=" * 80)

In [ ]:
import re
import time
import random
import pandas as pd
from youtube_transcript_api import YouTubeTranscriptApi

# =========================
# 설정
# =========================
IN_XLSX  = r"C:\py_temp\2차프로젝트\2026-01-14-11-31-49-유튜브_애니성지순례\2026-01-14-11-31-49-유튜브_애니성지순례_with_transcript.xlsx"
OUT_XLSX = r"C:\py_temp\2차프로젝트\2026-01-14-11-31-49-유튜브_애니성지순례\2026-01-14-11-31-49-유튜브_애니성지순례_with_transcript.xlsx"

NO_COL  = "번호"
URL_COL = "영상 링크"

PREFER_LANGS = ("ko", "en")

# ✅ 지금 29까지 완료라면 30부터 시작
START_NO = 320

# ✅ 운영 파라미터
BATCH_LIMIT = 60      # 이번 실행에서 최대 몇 개 처리할지
MIN_SLEEP = 150
MAX_SLEEP = 180
SAVE_EVERY = 5

# (선택) 테스트 URL 비우면 첫 대상 URL로 테스트
TEST_URL = "https://www.youtube.com/watch?v=6jycUExHOVc&pp=ygUZ7Iqs656o642p7YGsIOyEseyngOyInOuhgA%3D%3D"

# 실패 시 transcript에 넣을 문구(빈칸 방지)
FAIL_TEXT = "자막 추출이 불가능합니다"

# =========================
# 유틸/자막 함수 (v1.2.3 방식)
# =========================
def extract_video_id(url: str) -> str:
    m = re.search(r"(?:v=|youtu\.be/|embed/)([A-Za-z0-9_-]{11})", str(url))
    if not m:
        raise ValueError(f"영상 ID 추출 실패: {url}")
    return m.group(1)

def fetch_transcript_text_v123(url: str, prefer_langs=("ko", "en"), preserve_formatting=False) -> str:
    video_id = extract_video_id(url)
    api = YouTubeTranscriptApi()
    fetched = api.fetch(
        video_id,
        languages=list(prefer_langs),
        preserve_formatting=preserve_formatting
    )
    return "\n".join(snippet.text for snippet in fetched)

def is_rate_limited(msg: str) -> bool:
    s = (msg or "").lower()
    return ("ipblocked" in s) or ("too many requests" in s) or ("http error 429" in s) or (" 429" in s)

def safe_try_one(url: str):
    try:
        text = fetch_transcript_text_v123(url, prefer_langs=PREFER_LANGS)
        if text and text.strip():
            return True, text
        return False, "EMPTY_TEXT"
    except Exception as e:
        return False, f"{type(e).__name__}: {str(e)[:220]}"

def is_blank(x) -> bool:
    return (x is None) or (str(x).strip() == "") or (str(x).strip().lower() in ["nan", "none"])

# =========================
# 0) 엑셀 로드 + 컬럼 준비
# =========================
df = pd.read_excel(IN_XLSX)

if NO_COL not in df.columns:
    raise ValueError(f"엑셀에 '{NO_COL}' 컬럼이 없습니다.")
if URL_COL not in df.columns:
    raise ValueError(f"엑셀에 '{URL_COL}' 컬럼이 없습니다.")

df[NO_COL] = pd.to_numeric(df[NO_COL], errors="coerce")

# URL 정리
df[URL_COL] = df[URL_COL].astype(str)
df.loc[df[URL_COL].str.lower().isin(["nan", "none", ""]), URL_COL] = None

# transcript 컬럼 없으면 생성
if "transcript" not in df.columns:
    df["transcript"] = ""

# (선택) 로그용 컬럼
if "transcript_ok" not in df.columns:
    df["transcript_ok"] = None
if "transcript_err" not in df.columns:
    df["transcript_err"] = ""

# 번호 기준 정렬
df = df.sort_values(by=NO_COL, ascending=True).reset_index(drop=True)

# =========================
# ✅ 1) 이번 실행 대상 만들기
# - START_NO 이상
# - URL 존재
# - transcript가 빈칸인 것만 (성공/실패 상관없이 한번 채워지면 스킵)
# =========================
candidates = df[
    (df[NO_COL].notna()) &
    (df[NO_COL] >= START_NO) &
    (df[URL_COL].notna()) &
    (df["transcript"].apply(is_blank))
].copy()

print("✅ 남은 대상(빈 transcript):", len(candidates))

if len(candidates) == 0:
    print("✅ 이미 다 채워졌거나 START_NO 이후 빈 transcript가 없습니다.")
    df.to_excel(OUT_XLSX, index=False)
    raise SystemExit

# 이번 배치
candidates = candidates.head(BATCH_LIMIT)
target_idx = candidates.index.tolist()

print("🚀 이번 배치 처리:", len(target_idx))
print("  시작 번호:", int(df.at[target_idx[0], NO_COL]))

# =========================
# 2) 단일 테스트
# =========================
test_url = TEST_URL.strip() if TEST_URL.strip() else df.at[target_idx[0], URL_COL]
print("🧪 단일 테스트 URL:", test_url)

ok, res = safe_try_one(test_url)
if not ok:
    print("❌ 테스트 실패:", res)
    if is_rate_limited(res):
        print("➡️ 429/IpBlocked 계열. 쉬었다가 다시 테스트하세요.")
        raise SystemExit
    print("➡️ 차단은 아님(자막 없음/비공개/제한 가능). 그래도 진행하려면 TEST_URL 바꿔 테스트.")
    raise SystemExit

print("✅ 테스트 성공! (앞 120자)")
print(res[:120].replace("\n", " "))
time.sleep(2)

# =========================
# 3) 본 수집
# - 성공: transcript=자막
# - 실패: transcript=FAIL_TEXT (빈칸 방지)
# - 429: 저장하고 종료 (해당 번호 transcript는 비워둬서 다음에 그 번호부터 다시)
# =========================
success_cnt = 0
fail_cnt = 0

for k, idx in enumerate(target_idx, 1):
    no_val = int(df.at[idx, NO_COL])
    url = df.at[idx, URL_COL]

    print(f"\n[{k}/{len(target_idx)}] 번호={no_val}")
    print("  URL:", url)

    try:
        text = fetch_transcript_text_v123(url, prefer_langs=PREFER_LANGS)

        if text and text.strip():
            df.at[idx, "transcript"] = text
            df.at[idx, "transcript_ok"] = True
            df.at[idx, "transcript_err"] = ""
            success_cnt += 1
            print("  ✅ 성공 | 글자수:", len(text))
        else:
            # 실패여도 transcript를 채워서 '빈칸'이 아니게
            df.at[idx, "transcript"] = FAIL_TEXT
            df.at[idx, "transcript_ok"] = False
            df.at[idx, "transcript_err"] = "EMPTY_TEXT"
            fail_cnt += 1
            print("  ❌ 실패 | EMPTY_TEXT → transcript에 실패문구 저장")

    except Exception as e:
        err = f"{type(e).__name__}: {str(e)[:220]}"
        print("  ❌ 실패 |", err)

        if is_rate_limited(err):
            print(f"  🛑 429/IpBlocked → 번호={no_val}에서 저장 후 종료 (다음 실행은 {no_val}부터)")
            df.to_excel(OUT_XLSX, index=False)
            print("💾 저장:", OUT_XLSX)
            raise SystemExit

        # 429가 아니면 실패문구로 transcript 채워서 다음에 건드리지 않게
        df.at[idx, "transcript"] = FAIL_TEXT
        df.at[idx, "transcript_ok"] = False
        df.at[idx, "transcript_err"] = err
        fail_cnt += 1

    # 중간 저장
    if k % SAVE_EVERY == 0:
        df.to_excel(OUT_XLSX, index=False)
        print("  💾 중간 저장:", OUT_XLSX)

    # 랜덤 대기
    sleep_s = random.uniform(MIN_SLEEP, MAX_SLEEP)
    print(f"  ⏳ 대기 {sleep_s:.1f}s")
    time.sleep(sleep_s)

# 최종 저장
df.to_excel(OUT_XLSX, index=False)
print("\n✅ 배치 완료 | 성공:", success_cnt, "| 실패:", fail_cnt)
print("결과 파일:", OUT_XLSX)
print("다음 실행은 transcript가 빈칸인 번호부터 자동으로 이어서 진행합니다.")

In [36]:
import pandas as pd

# 1) 데이터 로드
df = pd.read_excel(
    "C:\\py_temp\\2차프로젝트\\2026-01-14-11-31-49-유튜브_애니성지순례\\2026-01-14-11-31-49-유튜브_애니성지순례_with_transcript.xls"
)

# NaN 방지
df["제목"] = df["제목"].fillna("")
df["transcript"] = df["transcript"].fillna("")

print("전체 영상 수:", len(df))

# 2) 성지 / 여행 키워드만 사용
travel_keywords = [
    "성지", "성지순례", "여행", "브이로그", "투어", "탐방", "순례"
]

# 3) 제목에 키워드 하나라도 포함되면 True
df["keep"] = df["제목"].apply(
    lambda title: any(k in title for k in travel_keywords)
)

# 4) 필터 적용
filtered_df = df[df["keep"] == True].drop(columns=["keep"]).reset_index(drop=True)

print("필터 후 영상 수:", len(filtered_df))

# =====================================================
# 🔥 여기 추가 (자막 정제 파트)
# =====================================================

# 줄바꿈 제거 (\n → 공백)
filtered_df["transcript"] = (
    filtered_df["transcript"]
    .astype(str)
    .str.replace(r"\\n", " ", regex=True)
    .str.replace(r"\s*\n\s*", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

filtered_df["내용"] = (
    filtered_df["내용"]
    .astype(str)
    .str.replace(r"\\n", " ", regex=True)
    .str.replace(r"\s*\n\s*", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# =====================================================
# 🔥 transcript_err 컬럼 제거
# =====================================================

if "transcript_err" in filtered_df.columns:
    filtered_df = filtered_df.drop(columns=["transcript_err"])

# =====================================================
# 5) 저장
# =====================================================

filtered_df.to_excel("youtube_final.xlsx", index=False)
print("저장 완료")

전체 영상 수: 320
필터 후 영상 수: 209
저장 완료


In [ ]:
youf = pd.read_excel("C:\\py_temp\\2차프로젝트\\2026-01-14-11-31-49-유튜브_애니성지순례\\youtube_final.xlsx")

youf_filtered = youf[
    youf["transcript"].notna() &                              # 1️⃣ NaN 제거
    (youf["transcript"].str.strip() != "") &                  # 2️⃣ 빈 문자열 제거
    (~youf["transcript"].str.contains("자막 추출이 불가능합니다", na=False)) &  # 3️⃣ 실패 문구 제거
    (~youf["transcript"].str.contains("자막을 불러올 수 없습니다", na=False)) &
    (~youf["transcript"].str.contains("TranscriptsDisabled", na=False)) &
    (~youf["transcript"].str.contains("Could not retrieve", na=False))
].reset_index(drop=True)

youf_filtered

In [ ]:
# ============================================================
# (def 없이 / openai 2.15.0 호환)
# 유튜브(내용+댓글+자막) -> 스팟 추출(JSON 강제 파싱) -> 문단 추출 -> 감성분석(JSON 강제 파싱)
# -> spot_sentiment_summary(종합행 포함) 저장
# ============================================================

import os
import re
import json
import time
import pandas as pd
from tqdm import tqdm
from openai import OpenAI

# ======================
# 0) 경로 설정
# ======================
IN_YT_PATH = r"C:\py_temp\2차프로젝트\2026-01-14-11-31-49-유튜브_애니성지순례\youtube_final.xlsx"   # 유튜브 전처리 파일
OUT_DIR    = r"C:\py_temp\2차프로젝트\유튜브_감성분석"               # 결과 저장 폴더

os.makedirs(OUT_DIR, exist_ok=True)

OUT_SPOT_CAND = os.path.join(OUT_DIR, "spot_candidates.xlsx")
OUT_MATCHED   = os.path.join(OUT_DIR, "matched_mentions.xlsx")
OUT_SUMMARY   = os.path.join(OUT_DIR, "spot_sentiment_summary.xlsx")
OUT_FAILLOG   = os.path.join(OUT_DIR, "json_fail_logs.xlsx")

# ======================
# 1) 컬럼 설정
# ======================
COL_ANIME      = "애니"
COL_TITLE      = "제목"
COL_URL        = "url"
COL_DESC       = "내용"
COL_COMMENT    = "1번 댓글"
COL_TRANSCRIPT = "transcript"

# ======================
# 2) API 키 직접 입력
# ======================
API_KEY = "sk"   # <- 여기만 채우기
client = OpenAI(api_key=API_KEY)

# ======================
# 3) 유튜브 DF 로드 + all_text 만들기
# ======================
yt_df = pd.read_excel(IN_YT_PATH)

if COL_URL not in yt_df.columns:
    yt_df[COL_URL] = ""

need_cols = [COL_ANIME, COL_TITLE, COL_DESC, COL_COMMENT, COL_TRANSCRIPT, COL_URL]
missing = [c for c in need_cols if c not in yt_df.columns]
if missing:
    raise KeyError(f"유튜브 DF에 컬럼이 없습니다: {missing}")

for c in need_cols:
    yt_df[c] = yt_df[c].fillna("").astype(str)

yt_df["all_text"] = (
    yt_df[COL_DESC].str.strip() + "\n" +
    yt_df[COL_COMMENT].str.strip() + "\n" +
    yt_df[COL_TRANSCRIPT].str.strip()
).str.strip()

print("rows:", len(yt_df))
print("all_text empty count:", (yt_df["all_text"].str.len().fillna(0) == 0).sum())

# ======================
# 4) JSON 파싱 보조 (정규식으로 { ... } 만 추출)
# ======================
def extract_json_block(text: str) -> str:
    if not text:
        return ""
    m = re.search(r"\{.*\}", text, flags=re.S)
    return m.group(0) if m else ""

# ======================
# 5) 스팟 추출 프롬프트
# ======================
SPOT_PROMPT = """
너는 텍스트에서 장소/지명/역/신사/사원/공원/거리/건물/상점 후보만 추출한다.
반드시 JSON만 출력하라. 다른 말 절대 금지.

형식:
{"spots":["도쿄역","신주쿠","타바타역"]}

규칙:
- 최대 30개
- 중복 제거
- 애매해도 후보로 포함
"""

# ======================
# 6) 감성분석 프롬프트
# ======================
SENTIMENT_PROMPT = """
너는 한국어 텍스트(여행/성지순례)의 감성분석기다.
반드시 JSON만 출력하라. 다른 말 절대 금지.

형식:
{"sentiment":"positive|neutral|negative","score":0.0,"confidence":0.0,"summary":"한줄 요약"}

규칙:
- score는 -1~1
- confidence는 0~1
"""

# ======================
# 7) 스팟 후보 추출(구버전 호환: JSON 강제 파싱 + 실패시 재요청 1회)
# ======================
spot_rows = []
fail_rows = []

for i, row in tqdm(yt_df.iterrows(), total=len(yt_df), desc="Spot extracting"):
    text = row["all_text"]
    if not text.strip():
        continue

    payload = text[:9000]

    # 1차 호출
    try:
        r = client.responses.create(
            model="gpt-4o-mini",
            input=[
                {"role":"system","content":SPOT_PROMPT},
                {"role":"user","content":payload},
            ],
        )
        raw = r.output_text or ""
        jtxt = extract_json_block(raw)

        # JSON 파싱 시도
        try:
            data = json.loads(jtxt)
        except:
            # 2차 재요청: JSON만 다시!
            r2 = client.responses.create(
                model="gpt-4o-mini",
                input=[
                    {"role":"system","content":"오직 JSON만 출력해라. 다른 말 금지."},
                    {"role":"user","content":f"아래 내용을 '{SPOT_PROMPT}' 형식의 JSON으로만 다시 출력해.\n\n[모델 출력]\n{raw[:1500]}"},
                ],
            )
            raw2 = r2.output_text or ""
            jtxt2 = extract_json_block(raw2)
            data = json.loads(jtxt2)

        spots = data.get("spots", [])
        if not isinstance(spots, list):
            # 문자열 등 이상 타입이면 분해
            if isinstance(spots, str):
                spots = [x.strip() for x in re.split(r"[,\n/]+", spots) if x.strip()]
            else:
                spots = []

        # 중복 제거 + 30개 제한
        spots = list(dict.fromkeys([str(s).strip() for s in spots if str(s).strip()]))[:30]

        if not spots:
            fail_rows.append([i, "spots_empty", row[COL_TITLE], (raw[:200] if raw else "")])
            continue

        for s in spots:
            spot_rows.append({
                "anime": row[COL_ANIME],
                "video_title": row[COL_TITLE],
                "video_url": row[COL_URL],
                "spot": s
            })

    except Exception as e:
        fail_rows.append([i, "spot_api_error", row[COL_TITLE], str(e)[:200]])
        time.sleep(0.5)
        continue

spot_cand_df = pd.DataFrame(spot_rows).drop_duplicates()
spot_cand_df.to_excel(OUT_SPOT_CAND, index=False)
print("✅ spot_candidates rows:", len(spot_cand_df))
print("✅ 저장:", OUT_SPOT_CAND)

# 실패 로그 저장(있으면)
fail_df = pd.DataFrame(fail_rows, columns=["row_idx","reason","title","detail"])
fail_df.to_excel(OUT_FAILLOG, index=False)
print("⚠️ json_fail_logs rows:", len(fail_df))
print("✅ 저장:", OUT_FAILLOG)

# ======================
# 8) (영상,spot) 조합만 문단 추출
# ======================
WINDOW = 2
MAX_HITS_PER_VIDEO = 20

video_text_map = {}
for _, row in yt_df.iterrows():
    key = (row[COL_ANIME], row[COL_TITLE], row[COL_URL])
    video_text_map[key] = row["all_text"]

matched_rows = []

for _, r in tqdm(spot_cand_df.iterrows(), total=len(spot_cand_df), desc="Context extracting"):
    anime = r["anime"]
    title = r["video_title"]
    url   = r["video_url"]
    spot  = r["spot"]

    all_text = video_text_map.get((anime, title, url), "")
    if not all_text.strip():
        continue

    lines = re.split(r"\r?\n+", all_text)
    lines = [ln.strip() for ln in lines if ln and ln.strip()]

    hits = 0
    for idx, ln in enumerate(lines):
        if spot in ln:
            start = max(0, idx - WINDOW)
            end   = min(len(lines), idx + WINDOW + 1)
            chunk = " ".join(lines[start:end]).strip()
            if chunk:
                matched_rows.append({
                    "anime": anime,
                    "spot": spot,
                    "video_title": title,
                    "video_url": url,
                    "matched_text": chunk
                })
            hits += 1
            if hits >= MAX_HITS_PER_VIDEO:
                break

matched_df = pd.DataFrame(matched_rows)
if not matched_df.empty:
    matched_df = matched_df.drop_duplicates(subset=["anime","spot","video_url","matched_text"])

matched_df.to_excel(OUT_MATCHED, index=False)
print("✅ matched_mentions(감성 전) rows:", len(matched_df))
print("✅ 저장:", OUT_MATCHED)

# ======================
# 9) 감성분석(JSON 강제 파싱 + 실패시 재요청 1회)
# ======================
if matched_df.empty:
    print("⚠️ 매칭된 문단이 없어 요약표는 빈 파일로 저장합니다.")
    pd.DataFrame(columns=["anime","spot","avg_score","pos_ratio","n_mentions"]).to_excel(OUT_SUMMARY, index=False)
    print("✅ 저장:", OUT_SUMMARY)
else:
    matched_df["sentiment"] = ""
    matched_df["score"] = 0.0
    matched_df["confidence"] = 0.0
    matched_df["summary"] = ""

    senti_fail = []

    for i, row in tqdm(matched_df.iterrows(), total=len(matched_df), desc="Sentiment"):
        payload = row["matched_text"][:6000]

        try:
            r = client.responses.create(
                model="gpt-4o-mini",
                input=[
                    {"role":"system","content":SENTIMENT_PROMPT},
                    {"role":"user","content":payload},
                ],
            )
            raw = r.output_text or ""
            jtxt = extract_json_block(raw)

            try:
                data = json.loads(jtxt)
            except:
                r2 = client.responses.create(
                    model="gpt-4o-mini",
                    input=[
                        {"role":"system","content":"오직 JSON만 출력해라. 다른 말 금지."},
                        {"role":"user","content":f"아래 내용을 '{SENTIMENT_PROMPT}' 형식의 JSON으로만 다시 출력해.\n\n[모델 출력]\n{raw[:1500]}"},
                    ],
                )
                raw2 = r2.output_text or ""
                jtxt2 = extract_json_block(raw2)
                data = json.loads(jtxt2)

            matched_df.loc[i, "sentiment"]  = data.get("sentiment", "neutral")
            matched_df.loc[i, "score"]      = float(data.get("score", 0.0) or 0.0)
            matched_df.loc[i, "confidence"] = float(data.get("confidence", 0.0) or 0.0)
            matched_df.loc[i, "summary"]    = data.get("summary", "")

        except Exception as e:
            matched_df.loc[i, "sentiment"] = "error"
            matched_df.loc[i, "summary"] = f"(error) {str(e)[:150]}"
            senti_fail.append([i, "sentiment_api_error", str(e)[:200]])
            time.sleep(0.3)

    matched_df.to_excel(OUT_MATCHED, index=False)
    print("✅ matched_mentions(감성 후) 저장:", OUT_MATCHED)

    # ======================
    # 10) 요약표 + 종합행
    # ======================
    matched_df["is_pos"] = (matched_df["sentiment"] == "positive").astype(int)

    summary_df = matched_df.groupby(["anime","spot"]).agg(
        avg_score=("score","mean"),
        pos_ratio=("is_pos","mean"),
        n_mentions=("score","count")
    ).reset_index()

    total_df = matched_df.groupby(["anime"]).agg(
        avg_score=("score","mean"),
        pos_ratio=("is_pos","mean"),
        n_mentions=("score","count")
    ).reset_index()
    total_df["spot"] = "ALL"
    total_df["anime"] = total_df["anime"].astype(str) + "(종합)"

    summary_df = pd.concat([summary_df, total_df], ignore_index=True)
    summary_df.to_excel(OUT_SUMMARY, index=False)
    print("✅ spot_sentiment_summary 저장:", OUT_SUMMARY)

print("\n끝!")

In [ ]:
IN_YT_PATH = r"C:\py_temp\2차프로젝트\2026-01-14-11-31-49-유튜브_애니성지순례\youtube_final.xlsx"   # 유튜브 전처리 파일
OUT_DIR    = r"C:\py_temp\2차프로젝트\유튜브_감성분석"               # 결과 저장 폴더

API_KEY = "sk"   # <- 여기만 채우기
client = OpenAI(api_key=API_KEY)

IN_SUMMARY = r"C:\py_temp\2차프로젝트\유튜브_감성분석\spot_sentiment_summary.xlsx"  # 유튜브 결과 요약표
IN_SPOTDB  = r"C:\py_temp\2차프로젝트\spot.xls"  # 민수님이 만든 스팟 키워드 파일
OUT_CLEAN  = r"C:\py_temp\2차프로젝트\유튜브_감성분석\spot_sentiment_summary_clean.xlsx"

In [ ]:
import os, re
import pandas as pd
from difflib import SequenceMatcher

# =========================
# 0) 경로 (이것만 바꾸면 됨)
# =========================
IN_MATCHED = r"C:\py_temp\2차프로젝트\유튜브_감성분석\matched_mentions.xlsx"   # 감성분석된 문단 파일
IN_SPOTDB  = r"C:\py_temp\2차프로젝트\spot.xls"  # 민수님이 만든 스팟 키워드 파일                
OUT_DIR    = r"C:\py_temp\2차프로젝트\유튜브_감성분석"

OUT_SUM_MAPPED = os.path.join(OUT_DIR, "spot_sentiment_summary_mapped.xlsx")
OUT_UNKNOWN    = os.path.join(OUT_DIR, "unknown_spots_rank.xlsx")

# =========================
# 1) 로드
# =========================
m = pd.read_excel(IN_MATCHED)
s = pd.read_excel(IN_SPOTDB)

# matched_mentions 필수컬럼 확인
need = ["anime","spot","sentiment","score"]
miss = [c for c in need if c not in m.columns]
if miss:
    raise KeyError(f"matched_mentions에 컬럼 없음: {miss} / 현재: {m.columns.tolist()}")

# spot db에서 스팟 컬럼 자동 선택 (민수님 파일에 맞춰 후보들 넣어둠)
cand_cols = ["spot"]
spot_col = None
for c in cand_cols:
    if c in s.columns:
        spot_col = c
        break
if spot_col is None:
    raise KeyError(f"스팟 키워드 파일에서 장소 컬럼을 못 찾음. 현재 컬럼: {s.columns.tolist()}")

# =========================
# 2) 정규화 컬럼 만들기(공백/기호 제거 + 자주 붙는 접두/접미 제거)
# =========================
m["spot_raw"] = m["spot"].fillna("").astype(str).str.strip()
s["spot_std"] = s[spot_col].fillna("").astype(str).str.strip()

# 빈 값 제거
s = s[s["spot_std"] != ""].copy()

# 정규화 함수 대신 "inline" 처리 (def 없이)
# - 공백 제거
# - 괄호/특수문자 제거
# - 'JR', '역', 'Station', '점', '출구' 같은 군더더기 제거(너무 과하면 오탐 늘 수 있어 조절 가능)
def_cleanup = [
    (r"\s+", ""),                 # 모든 공백 제거
    (r"[\(\)\[\]\{\}]", ""),      # 괄호 제거
    (r"[·•\-_/]", ""),            # 구분기호 제거
    (r"(?i)station", ""),         # station 제거(영문)
    (r"(?i)jr", ""),              # JR 제거(영문)
    (r"역$", ""),                 # 끝이 '역'이면 제거
    (r"점$", ""),                 # 끝이 '점'이면 제거(매장)
    (r"(동쪽|서쪽|남쪽|북쪽)?출구$", ""),  # 출구 제거
]

m["spot_norm"] = m["spot_raw"]
s["spot_norm"] = s["spot_std"]

for pat, rep in def_cleanup:
    m["spot_norm"] = m["spot_norm"].str.replace(pat, rep, regex=True)
    s["spot_norm"] = s["spot_norm"].str.replace(pat, rep, regex=True)

# 사전 중복 제거
s = s.drop_duplicates(subset=["spot_norm","spot_std"]).copy()

# 사전 리스트 준비
dict_norm = s["spot_norm"].tolist()
dict_std  = s["spot_std"].tolist()

# =========================
# 3) 매핑(붙이기): exact -> contains -> fuzzy(SequenceMatcher)
# =========================
THRESH = 0.78   # fuzzy 매칭 임계치(0.75~0.85 사이로 조절 추천)
USE_FUZZY = True

mapped_std = []
mapped_rule = []

for sn in m["spot_norm"].tolist():
    sn = (sn or "").strip()
    if sn == "":
        mapped_std.append("")
        mapped_rule.append("empty")
        continue

    # (A) exact match
    if sn in s["spot_norm"].values:
        # 동일 norm인 첫 표준명 사용
        mapped = s.loc[s["spot_norm"] == sn, "spot_std"].iloc[0]
        mapped_std.append(mapped)
        mapped_rule.append("exact")
        continue

    # (B) contains match (사전키가 문장에 포함 or 역으로 포함)
    hit_idx = None
    for j, dn in enumerate(dict_norm):
        if dn and (dn in sn or sn in dn):
            hit_idx = j
            break

    if hit_idx is not None:
        mapped_std.append(dict_std[hit_idx])
        mapped_rule.append("contains")
        continue

    # (C) fuzzy match
    if USE_FUZZY:
        best_score = 0.0
        best_idx = None
        for j, dn in enumerate(dict_norm):
            if not dn:
                continue
            score = SequenceMatcher(None, sn, dn).ratio()
            if score > best_score:
                best_score = score
                best_idx = j

        if best_idx is not None and best_score >= THRESH:
            mapped_std.append(dict_std[best_idx])
            mapped_rule.append(f"fuzzy_{best_score:.2f}")
            continue

    # (D) unknown
    mapped_std.append("UNKNOWN")
    mapped_rule.append("unknown")

m["spot_mapped"] = mapped_std
m["map_rule"] = mapped_rule

print("매핑 규칙 분포:\n", m["map_rule"].value_counts().head(20))
print("UNKNOWN 개수:", (m["spot_mapped"]=="UNKNOWN").sum())

# =========================
# 4) 요약표 만들기(애니×대표스팟) + 종합(ALL)
# =========================
m["is_pos"] = (m["sentiment"].astype(str).str.lower() == "positive").astype(int)

# UNKNOWN 제외한 깨끗한 요약
use = m[m["spot_mapped"] != "UNKNOWN"].copy()

summary = use.groupby(["anime","spot_mapped"], as_index=False).agg(
    avg_score=("score","mean"),
    pos_ratio=("is_pos","mean"),
    n_mentions=("score","count"),
)

summary = summary.rename(columns={"spot_mapped":"spot"})

# 애니별 종합행
total = use.groupby(["anime"], as_index=False).agg(
    avg_score=("score","mean"),
    pos_ratio=("is_pos","mean"),
    n_mentions=("score","count"),
)
total["spot"] = "ALL"
total["anime"] = total["anime"].astype(str) + "(종합)"

out = pd.concat([summary, total], ignore_index=True)

out.to_excel(OUT_SUM_MAPPED, index=False)
print("✅ 저장:", OUT_SUM_MAPPED, "| rows:", len(out))

# =========================
# 5) UNKNOWN 후보 랭킹(사전에 없던 스팟=추가 정리할 리스트)
# =========================
unknown_rank = m[m["spot_mapped"]=="UNKNOWN"].groupby(["anime","spot_raw"], as_index=False).agg(
    n_mentions=("spot_raw","count"),
    avg_score=("score","mean"),
    pos_ratio=("is_pos","mean"),
).sort_values(["n_mentions","avg_score"], ascending=[False, False])

unknown_rank.to_excel(OUT_UNKNOWN, index=False)
print("✅ UNKNOWN 랭킹 저장:", OUT_UNKNOWN, "| rows:", len(unknown_rank))

In [ ]:
import pandas as pd
import re

IN_UNKNOWN = r"C:\py_temp\2차프로젝트\유튜브_감성분석\unknown_spots_rank.xlsx"
OUT_PLACE_ONLY = r"C:\py_temp\2차프로젝트\유튜브_감성분석\unknown_spots_rank_place_only.xlsx"

u = pd.read_excel(IN_UNKNOWN).copy()

# 1) 기본 전처리
u["anime"] = u["anime"].fillna("").astype(str).str.strip()
u["spot_raw"] = u["spot_raw"].fillna("").astype(str).str.strip()

# 2) 애니명과 동일/포함이면 제거 (ex: 주술회전 -> 주술회전)
mask_anime_same = (u["spot_raw"] == u["anime"])
mask_anime_in   = u.apply(lambda r: (r["anime"] != "" and r["anime"] in r["spot_raw"]), axis=1)
u2 = u[~(mask_anime_same | mask_anime_in)].copy()

# 3) 일반명사(노이즈) 제거 리스트
ban_words = [
    "계단","골목","바다","하늘","산","강","길","거리","도시","마을",
    "여행","성지","순례","코스","장면","영상","음악","감독","작품",
    "애니","영화","브이로그","리뷰","소개","먹방",
    "신칸센","전철","기차","버스","지하철","공항"
]
u2 = u2[~u2["spot_raw"].isin(ban_words)].copy()

# 4) “장소처럼 보이는 패턴”만 우선 남기기 (느슨하게)
#    - 완전 지명(가마쿠라)도 살려야 해서 '패턴 OR 지명(한글 2~6자)'도 허용
place_suffix = r"(역|신사|사원|공원|타워|거리|광장|시장|학교|대학|항|항구|비치|해변|다리)$"
mask_suffix = u2["spot_raw"].str.contains(place_suffix, regex=True)

mask_hangul_place = u2["spot_raw"].str.match(r"^[가-힣]{2,8}$")  # 가마쿠라/에노시마/하코다테 같은 순수 지명 살리기

u_place = u2[mask_suffix | mask_hangul_place].copy()

# 5) 언급수 필터(원하면 조절)
u_place = u_place[u_place["n_mentions"] >= 3].copy()
u_place = u_place.sort_values(["n_mentions","avg_score"], ascending=[False, False])

u_place.to_excel(OUT_PLACE_ONLY, index=False)
print("✅ 장소 후보만 저장:", OUT_PLACE_ONLY, "| rows:", len(u_place))
u_place.head(30)

In [ ]:
import pandas as pd

IN_UNKNOWN = r"C:\py_temp\2차프로젝트\유튜브_감성분석\unknown_spots_rank.xlsx"
OUT_TK_HINT = r"C:\py_temp\2차프로젝트\유튜브_감성분석\unknown_spots_tokyo_kyoto_hint.xlsx"

u = pd.read_excel(IN_UNKNOWN)

keywords = ["도쿄","신주쿠","시부야","아키하바라","우에노","아사쿠사","긴자","하라주쿠","이케부쿠로",
            "도쿄역","메이지","요요기","교토","기요미즈","후시미","아라시야마","가와라마치","니시키"]

mask = u["spot_raw"].astype(str).apply(lambda x: any(k in x for k in keywords))
cand = u[mask].sort_values(["n_mentions","avg_score"], ascending=[False, False]).copy()

cand.to_excel(OUT_TK_HINT, index=False)
print("✅ 저장:", OUT_TK_HINT, "| rows:", len(cand))
cand.head(50)

In [ ]:
import pandas as pd
import re

IN_TK = r"C:\py_temp\2차프로젝트\유튜브_감성분석\unknown_spots_tokyo_kyoto_hint.xlsx"
OUT_TK_MERGED = r"C:\py_temp\2차프로젝트\유튜브_감성분석\tokyo_kyoto_hint_merged.xlsx"

df = pd.read_excel(IN_TK).copy()
df["spot_raw"] = df["spot_raw"].fillna("").astype(str).str.strip()

# 1) 기본 정규화(공백/기호 제거)
spot_norm = df["spot_raw"].str.replace(r"\s+", "", regex=True)
spot_norm = spot_norm.str.replace(r"[·•\-_／/]", "", regex=True)

# 2) 자주 나오는 오타/표기 통일(필요한 만큼만 추가)
spot_norm = spot_norm.str.replace("아이랜드", "아일랜드", regex=False)
spot_norm = spot_norm.str.replace("타워", "타워", regex=False)  # 예시(추가 치환 가능)

df["spot_norm"] = spot_norm

# 3) spot_norm 기준으로 묶어서 합치기
merged = df.groupby(["anime", "spot_norm"], as_index=False).agg(
    spot_examples=("spot_raw", lambda x: " | ".join(pd.unique(x)[:5])),
    n_mentions=("n_mentions", "sum"),
    avg_score=("avg_score", "mean"),
    pos_ratio=("pos_ratio", "mean")
).sort_values(["n_mentions","avg_score"], ascending=[False, False])

merged.to_excel(OUT_TK_MERGED, index=False)
print("✅ 저장:", OUT_TK_MERGED, "| rows:", len(merged))
merged.head(30)

In [ ]:
import os
import pandas as pd

# =========================
# 0) 입력/출력 경로
# =========================
IN_MAPPED = r"C:\py_temp\2차프로젝트\유튜브_감성분석\spot_sentiment_summary_mapped.xlsx"
IN_UNK_TK = r"C:\py_temp\2차프로젝트\유튜브_감성분석\tokyo_kyoto_hint_merged.xlsx"

OUT_FINAL = r"C:\py_temp\2차프로젝트\유튜브_감성분석\spot_sentiment_summary_FINAL.xlsx"

# =========================
# 1) 로드
# =========================
m = pd.read_excel(IN_MAPPED)
u = pd.read_excel(IN_UNK_TK)

# =========================
# 2) 형태 맞추기
#   - mapped: [anime, spot, avg_score, pos_ratio, n_mentions] + (종합행 ALL)
#   - unknown merged: [anime, spot_norm, ..., n_mentions, avg_score, pos_ratio]
# =========================
# unknown쪽을 요약표 형태로 맞춤
u2 = u[["anime", "spot_norm", "avg_score", "pos_ratio", "n_mentions"]].copy()
u2 = u2.rename(columns={"spot_norm": "spot"})

# mapped쪽에서 종합행(spot=ALL)은 잠시 빼고 합치기(중복 계산 방지)
m_base = m[m["spot"].astype(str).str.upper() != "ALL"].copy()

# =========================
# 3) 합치기(Concat)
# =========================
cat = pd.concat([m_base, u2], ignore_index=True)

# 타입 정리
cat["anime"] = cat["anime"].fillna("").astype(str).str.strip()
cat["spot"] = cat["spot"].fillna("").astype(str).str.strip()

for c in ["avg_score", "pos_ratio", "n_mentions"]:
    cat[c] = pd.to_numeric(cat[c], errors="coerce")

cat = cat.dropna(subset=["anime", "spot", "n_mentions"]).copy()
cat = cat[cat["anime"] != ""]
cat = cat[cat["spot"] != ""]

# =========================
# 4) (anime, spot) 기준으로 재집계
#   - avg_score / pos_ratio: n_mentions 가중평균
#   - n_mentions: 합
# =========================
cat["w_score"] = cat["avg_score"] * cat["n_mentions"]
cat["w_pos"]   = cat["pos_ratio"] * cat["n_mentions"]

final_spot = cat.groupby(["anime", "spot"], as_index=False).agg(
    n_mentions=("n_mentions", "sum"),
    w_score=("w_score", "sum"),
    w_pos=("w_pos", "sum"),
)

final_spot["avg_score"] = final_spot["w_score"] / final_spot["n_mentions"]
final_spot["pos_ratio"] = final_spot["w_pos"] / final_spot["n_mentions"]

final_spot = final_spot.drop(columns=["w_score", "w_pos"])

# =========================
# 5) 애니별 종합행(ALL) 다시 생성
# =========================
final_spot["w_score2"] = final_spot["avg_score"] * final_spot["n_mentions"]
final_spot["w_pos2"]   = final_spot["pos_ratio"] * final_spot["n_mentions"]

total = final_spot.groupby(["anime"], as_index=False).agg(
    n_mentions=("n_mentions", "sum"),
    w_score2=("w_score2", "sum"),
    w_pos2=("w_pos2", "sum"),
)

total["avg_score"] = total["w_score2"] / total["n_mentions"]
total["pos_ratio"] = total["w_pos2"] / total["n_mentions"]
total["spot"] = "ALL"
total["anime"] = total["anime"].astype(str) + "(종합)"

total = total[["anime", "spot", "avg_score", "pos_ratio", "n_mentions"]].copy()

final_spot = final_spot.drop(columns=["w_score2", "w_pos2"])

# =========================
# 6) 최종 저장
# =========================
out = pd.concat([final_spot, total], ignore_index=True)
out = out[["anime", "spot", "avg_score", "pos_ratio", "n_mentions"]].copy()

out.to_excel(OUT_FINAL, index=False)
print("✅ 최종 저장:", OUT_FINAL, "| rows:", len(out))

# 상위 확인(언급수 큰 순)
print("\n[TOP 20 by n_mentions]")
print(out[out["spot"]!="ALL"].sort_values(["n_mentions","avg_score"], ascending=[False, False]).head(20))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# 0) 파일 경로
# =========================
IN_SUMMARY = r"C:\py_temp\2차프로젝트\유튜브_감성분석\spot_sentiment_summary_FINAL.xlsx"

plt.rcParams["font.family"] = "Malgun Gothic"   # 윈도우 기본 한글폰트
plt.rcParams["axes.unicode_minus"] = False      # 마이너스 깨짐 방지

# =========================
# 1) 데이터 로드
# =========================
df = pd.read_excel(IN_SUMMARY)

# =========================
# 2) 대상 애니만 필터
# =========================
targets = ["주술회전", "명탐정코난"]
df2 = df[df["anime"].isin(targets)].copy()

# 종합행 제거
df2 = df2[df2["spot"] != "ALL"].copy()

# =========================
# 3) TOP 스팟 정렬
# =========================
df2 = df2.sort_values(["anime","n_mentions","avg_score"], ascending=[True, False, False])

top_df = df2.groupby("anime").head(10).copy()

print("=== TOP 스팟 테이블 ===")
print(top_df)

# =========================
# 4) 애니별 막대그래프
# =========================
for anime in targets:
    sub = top_df[top_df["anime"] == anime]
    
    plt.figure()
    plt.barh(sub["spot"], sub["avg_score"])
    plt.title(f"{anime} - 성지 스팟 감성 점수 TOP")
    plt.xlabel("avg_score")
    plt.gca().invert_yaxis()
    plt.show()

In [ ]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from wordcloud import WordCloud

# =========================
# 0) 경로
# =========================
IN_XLSX = r"C:\py_temp\2차프로젝트\유튜브_감성분석\spot_sentiment_summary_FINAL.xlsx"
OUT_DIR = r"C:\py_temp\2차프로젝트\유튜브_감성분석\wordcloud_by_anime"
os.makedirs(OUT_DIR, exist_ok=True)

# =========================
# 1) 한글 폰트 (Windows)
# =========================
FONT_PATH = r"C:\Windows\Fonts\malgun.ttf"
if not os.path.exists(FONT_PATH):
    alt = r"C:\Windows\Fonts\NanumGothic.ttf"
    FONT_PATH = alt if os.path.exists(alt) else None
if FONT_PATH is None:
    raise FileNotFoundError("한글 폰트를 못 찾았어. malgun.ttf 또는 NanumGothic.ttf 확인해줘.")

# =========================
# 2) 로드
# =========================
df = pd.read_excel(IN_XLSX)
print("rows:", len(df))
print("columns:", list(df.columns))

# =========================
# 3) 컬럼 자동 탐색
# =========================
anime_candidates  = ["anime", "애니", "작품", "title", "anime_title", "anime_nm"]
spot_candidates   = ["spot_norm", "spot", "스팟", "장소", "place", "spot_name", "spot_examples"]
weight_candidates = ["n_mentions", "mentions", "cnt", "count", "freq", "언급", "빈도"]

anime_col = next((c for c in anime_candidates if c in df.columns), None)
spot_col  = next((c for c in spot_candidates if c in df.columns), None)
w_col     = next((c for c in weight_candidates if c in df.columns), None)

if anime_col is None:
    raise ValueError(f"애니 컬럼을 못 찾았어. 현재 컬럼들: {list(df.columns)}\n"
                     f"anime_candidates에 실제 컬럼명을 추가해줘.")
if spot_col is None:
    raise ValueError(f"스팟 컬럼을 못 찾았어. 현재 컬럼들: {list(df.columns)}\n"
                     f"spot_candidates에 실제 컬럼명을 추가해줘.")

print("✅ anime_col:", anime_col)
print("✅ spot_col :", spot_col)
print("✅ weight_col:", w_col if w_col else "(없음 → 모두 1로 처리)")

# =========================
# 요약(ALL / 종합) 행 제거
# =========================
df = df[
    (~df[anime_col].astype(str).str.upper().eq("ALL")) &
    (~df[anime_col].astype(str).str.contains("종합", na=False))
].copy()

print("남은 애니 목록:")
print(df[anime_col].unique())

# =========================
# 4) 파일명 안전화
# =========================
def safe_name(s: str) -> str:
    s = str(s).strip()
    s = re.sub(r'[\\/:*?"<>|]', "_", s)  # 윈도우 금지문자 치환
    s = re.sub(r"\s+", "_", s)
    return s[:80] if len(s) > 80 else s

# =========================
# 5) 애니 목록 확인 (7개만 만들기)
# =========================
anime_list = (
    df[anime_col].dropna().astype(str).str.strip()
    .loc[lambda x: x != ""]
    .unique()
    .tolist()
)

print(f"✅ 애니 고유 개수: {len(anime_list)}")
print("✅ 애니 목록:", anime_list)

# 혹시 7개보다 많으면, 언급합(n_mentions 합) 큰 상위 7개로 자르기
if len(anime_list) > 7 and w_col is not None:
    top7 = (df.groupby(anime_col, dropna=True)[w_col].sum()
            .sort_values(ascending=False).head(7).index.astype(str).tolist())
    anime_list = top7
    print("⚠️ 애니가 7개 초과라서, 언급합 기준 TOP 7만 생성:", anime_list)
else:
    # 7개보다 적으면 있는 만큼만 생성
    anime_list = anime_list[:7]

# =========================
# 6) 애니별 워드클라우드 생성
# =========================
for anime_name in anime_list:
    sub = df[df[anime_col].astype(str).str.strip() == str(anime_name).strip()].copy()
    if sub.empty:
        print("skip (empty):", anime_name)
        continue

    freq = Counter()

    for _, row in sub.iterrows():
        s = row.get(spot_col, "")
        if pd.isna(s):
            continue
        s = str(s).strip()
        if not s:
            continue

        w = 1
        if w_col is not None:
            try:
                w = float(row.get(w_col, 1))
                if pd.isna(w):
                    w = 1
            except:
                w = 1

        # "A | B | C" 형태면 분해해서 각각 가중치 반영
        parts = [p.strip() for p in s.replace("｜", "|").replace(",", "|").split("|")]
        parts = [p for p in parts if p]

        for p in parts:
            freq[p] += w

    if len(freq) == 0:
        print("⚠️ 단어 없음:", anime_name)
        continue

    wc = WordCloud(
        font_path=FONT_PATH,
        width=1600,
        height=900,
        background_color="white",
        collocations=False
    ).generate_from_frequencies(dict(freq))

    out_png = os.path.join(OUT_DIR, f"wordcloud_{safe_name(anime_name)}.png")

    plt.figure(figsize=(14, 8))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(anime_name, fontsize=18)
    plt.tight_layout()
    plt.savefig(out_png, dpi=250, bbox_inches="tight")
    plt.show()
    plt.close()

    print("✅ 저장:", out_png)

print("\n끝! 폴더 확인:", OUT_DIR)